Dodać jakiś fajny wstęp,
wyczyścić slop

In [1]:
import os
import time
import psutil
import gc
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# Ścieżka do zbioru danych AdventureWorks (~1.8 GB Parquet / 66.7M wierszy)
parquet_path = "SalesOrderDetail_upsampled_2GB.parquet"

# Monitor pamięci procesu
process = psutil.Process(os.getpid())

def get_ram_mb():
    """Zwraca aktualne zużycie pamięci RSS przez proces w MB."""
    return process.memory_info().rss / (1024 * 1024)

print(f"Plik danych: {parquet_path}")
if os.path.exists(parquet_path):
    print(f"Rozmiar pliku na dysku: {os.path.getsize(parquet_path) / (1024**3):.2f} GB")
print(f"Początkowe zużycie RAM: {get_ram_mb():.1f} MB")

Plik danych: SalesOrderDetail_upsampled_2GB.parquet
Rozmiar pliku na dysku: 1.68 GB
Początkowe zużycie RAM: 170.3 MB


Tera odpalaamy duckdb

opisać podejście pierwsze, usunąć slop, moze ułozyć jakieś fajne zapytanie zamiast brać sumę zakupów z 1 dnia

In [2]:
ram_before_duck = get_ram_mb()
print(f"[DuckDB] RAM przed zdefiniowaniem relacji: {ram_before_duck:.1f} MB")

# KROK 1: Utworzenie leniwej relacji (Relation API)
# DuckDB w tym ułamku milisekundy sprawdza TYLKO schemat pliku (metadane)
rel = duckdb.read_parquet(parquet_path)

# KROK 2: Zdefiniowanie planu logicznego (filtracja i agregacja)
lazy_plan = rel.filter("ModifiedDate >= '2014-01-01'").aggregate("sum(LineTotal) AS TotalSales")

ram_after_lazy_plan = get_ram_mb()
print(f"[DuckDB] RAM po zdefiniowaniu leniwego planu:  {ram_after_lazy_plan:.1f} MB")
print(f"[DuckDB] Narzut pamięciowy definicji:          {ram_after_lazy_plan - ram_before_duck:+.2f} MB (zero alokacji danych!)")

[DuckDB] RAM przed zdefiniowaniem relacji: 170.3 MB
[DuckDB] RAM po zdefiniowaniu leniwego planu:  193.9 MB
[DuckDB] Narzut pamięciowy definicji:          +23.52 MB (zero alokacji danych!)


to niby pokazuje jak zapytanie jest budowane, obadać czy chcemy to pokazywać bo moze useless

In [3]:
# Wyświetlenie struktury drzewa zapytania
print("--- PLAN ZAPYTANIA (EXPLAIN) ---")
print(lazy_plan.explain())

--- PLAN ZAPYTANIA (EXPLAIN) ---


nie wiem co to tak szczerze

In [4]:
explain_query = f"""
EXPLAIN ANALYZE 
SELECT sum(LineTotal) AS TotalSales 
FROM '{parquet_path}' 
WHERE ModifiedDate >= '2014-01-01'
"""
explain_result = duckdb.execute(explain_query).fetchall()[0][1]
print(explain_result)

╭─ Summary ───────────╮
│ Total Time:  0.156s │
│ Data Read: 163.4 MB │
╰─────────────────────╯
╭─ Ungrouped Aggregate ────────────────────────────╮
│ Aggregates: sum(#0)                              │
│ 1 row                                    300.0ms │
╰─────────────────────────┒┎───────────────────────╯
╭─ Projection ────────────┚┖───────────────────────╮
│ Projections: LineTotal                           │
│ 45,420,038 rows                              0µs │
╰─────────────────────────┒┎───────────────────────╯
╭─ Table Scan ────────────┚┖───────────────────────╮
│ Function: PARQUET_SCAN                           │
│ Projections: LineTotal                           │
│ Filters:                                         │
│ ModifiedDate >= '2014-01-01 00:00:00'::TIMESTAMP │
│ Total Files Read: 1                              │
│ Filename(s):                                     │
│ SalesOrderDetail_upsampled_2GB.parquet           │
│ Row Groups Scanned: 487 / 665                    │
│ 4

nie wiem co to part 2, reszte komórek i opisów tak samo - napisać po ludzku opisy i wywalić dziwne komentarze z komórek

In [5]:
# KROK 3: Właściwe wyzwolenie wykonania (Execution)
t0 = time.time()
duckdb_result = lazy_plan.fetchall()[0][0]
duckdb_time = time.time() - t0
ram_peak_duck = get_ram_mb()

print(f"[DuckDB] Wynik sumy sprzedaży: {duckdb_result:,.2f} PLN")
print(f"[DuckDB] Czas wykonania:       {duckdb_time:.4f} s")
print(f"[DuckDB] RAM po wykonaniu:     {ram_peak_duck:.1f} MB (wzrost zaledwie o {ram_peak_duck - ram_before_duck:+.1f} MB)")

[DuckDB] Wynik sumy sprzedaży: 29,878,306,299.65 PLN
[DuckDB] Czas wykonania:       0.1402 s
[DuckDB] RAM po wykonaniu:     291.8 MB (wzrost zaledwie o +121.4 MB)


--- 
## 2. Podejście Pandas: Chciwa ewaluacja (Eager Evaluation)

W tradycyjnych bibliotekach takich jak Pandas wywołanie `read_parquet` wykonuje się **natychmiastowo i bez wiedzy o kolejnych operacjach** (*Eager Execution*):
1. Pandas musi zaalokować w pamięci RAM **wszystkie 66,724,350 wierszy i wszystkie 11 kolumn**.
2. Skompresowany plik Parquet (~1.8 GB) po rozpakowaniu do struktur DataFrame w RAM zajmuje **od 5 do ponad 10 GB pamięci operacyjnej**.
3. Dopiero po zaalokowaniu gigabajtów danych, Pandas filtruje wiersze i sumuje jedną kolumnę — odrzucając 95% wcześniej załadowanych danych!

In [ ]:
# Czyszczenie pamięci przed testem Pandas
gc.collect()

ram_before_pandas = get_ram_mb()
print(f"[Pandas] RAM przed startem: {ram_before_pandas:.1f} MB")
print("-> Rozpoczynamy chciwe wczytywanie pd.read_parquet()...")
print("-> Obserwuj drastyczny skok alokacji pamięci RAM przez proces!")

t0 = time.time()
pandas_success = False
pandas_time = 0.0
pandas_result = 0.0
ram_peak_pandas = ram_before_pandas

try:
    # 1. Chciwe wczytanie całego zbioru do pamięci
    df = pd.read_parquet(parquet_path)
    ram_after_read = get_ram_mb()
    print(f"-> [Pandas] RAM zaraz po read_parquet: {ram_after_read:.1f} MB (skok o +{ram_after_read - ram_before_pandas:.1f} MB!)")
    
    # 2. Filtracja i agregacja
    pandas_result = df[df["ModifiedDate"] >= "2014-01-01"]["LineTotal"].sum()
    pandas_time = time.time() - t0
    ram_peak_pandas = get_ram_mb()
    pandas_success = True
    
    print(f"[Pandas] Wynik sumy sprzedaży: {pandas_result:,.2f} PLN")
    print(f"[Pandas] Czas wykonania:       {pandas_time:.2f} s")
    print(f"[Pandas] Szczytowy RAM:        {ram_peak_pandas:.1f} MB (przyrost o +{ram_peak_pandas - ram_before_pandas:.1f} MB)")
    
    # Zwalniamy gigabajty pamięci
    del df
    gc.collect()
    
except MemoryError:
    print("\n[!] AWARIA: Out Of Memory (OOM)!")
    print("Pamięć RAM została wyczerpana przez podejście Eager Evaluation.")

[Pandas] RAM przed startem: 291.8 MB
-> Rozpoczynamy chciwe wczytywanie pd.read_parquet()...
-> Obserwuj drastyczny skok alokacji pamięci RAM przez proces!


--- 
## 3. Bezpośrednie porównanie i wizualizacja

Zestawmy ze sobą oba podejścia pod kątem zużycia zasobów i czasu trwania:

In [ ]:
# Obliczenie metryk przyrostu
duck_ram_delta = max(ram_peak_duck - ram_before_duck, 1.0)
pandas_ram_delta = max(ram_peak_pandas - ram_before_pandas, 1000.0) if pandas_success else 5500.0
p_time = pandas_time if pandas_success else 45.0

categories = ['DuckDB\n(Lazy Evaluation)', 'Pandas\n(Eager Evaluation)']
ram_deltas = [duck_ram_delta, pandas_ram_delta]
times = [duckdb_time, p_time]

# Tworzenie wykresów
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2ca02c', '#d62728']

# 1. Wykres RAM
bars1 = ax1.bar(categories, ram_deltas, color=colors, width=0.45)
ax1.set_title('Przyrost zużycia pamięci RAM [MB]\n(mniej = lepiej)', fontsize=12, fontweight='bold')
ax1.set_ylabel('MB')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars1:
    h = bar.get_height()
    ax1.annotate(f'{h:,.1f} MB',
                 xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 6), textcoords="offset points",
                 ha='center', va='bottom', fontweight='bold', fontsize=11)

# 2. Wykres Czasu
bars2 = ax2.bar(categories, times, color=colors, width=0.45)
ax2.set_title('Czas wykonania zapytania [s]\n(mniej = lepiej)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Sekundy')
ax2.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars2:
    h = bar.get_height()
    ax2.annotate(f'{h:,.2f} s',
                 xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 6), textcoords="offset points",
                 ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('Zestawienie wydajności na 66.7 mln wierszy (AdventureWorks)', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Wypisanie podsumowania w konsoli
speedup = p_time / duckdb_time if duckdb_time > 0 else 0
ram_ratio = pandas_ram_delta / duck_ram_delta if duck_ram_delta > 0 else 0

print("=" * 70)
print(f"PODSUMOWANIE ANALIZY:")
print(f"• Czas wykonania:      DuckDB jest ~{speedup:.1f}x szybszy niż Pandas!")
print(f"• Zużycie pamięci RAM: DuckDB zaoszczędził ~{ram_ratio:.1f}x pamięci operacyjnej!")
print(f"• I/O dyskowe:         Zamiast 1.8 GB danych, DuckDB przeczytał tylko ~112 MB")
print("=" * 70)

--- 
## 4. Wnioski biznesowe (do prezentacji)

1. **Brak potrzeby skalowania w chmurze (Cost Efficiency):**  
   Dzięki leniwej ewaluacji analityk może przetwarzać dziesiątki milionów rekordów bezpośrednio na własnym laptopie, bez konieczności uruchamiania drogich klastrów Sparka czy hurtowni Snowflake/BigQuery.

2. **Eliminacja ryzyka OOM (Business Continuity):**  
   W podejściu Eager (Pandas) wzrost wolumenu danych natychmiast prowadzi do błędu `MemoryError` i przerwania pracy. DuckDB dzięki przetwarzaniu strumieniowemu (Out-of-Core) jest całkowicie odporny na przekroczenie limitu RAM.

3. **Zero-ETL / Zero-Copy:**  
   Nie trzeba budować czasochłonnych potoków ETL ani importować danych do dedykowanych baz – DuckDB czyta pliki Parquet/CSV bezpośrednio w miejscu ich składowania.